In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as f
import torchvision
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

'''
1. 데이터 정규화 새로하기
2. epoch 늘리기
'''

### CIFAR10

In [ ]:
# CIFAR-10 데이터 다운로드 (train set만)
data_dir = "./data"

# 임시 전처리 (Resize + Tensor)
stats_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# 임시 데이터셋 로드
stats_dataset = datasets.CIFAR10(root=data_dir, train=True, download=True, transform=stats_transform)
stats_loader = DataLoader(stats_dataset, batch_size=64, shuffle=False)

# 채널별(RGB) 합계
# [R, G, B]
channels_sum = torch.zeros(3)

# 전체 픽셀 수
num_pixels = len(stats_dataset) * 224 * 224

# 픽셀 합 계산
for images, _ in stats_loader:
    channels_sum += torch.sum(images, dim=[0, 2, 3])

mean = channels_sum / num_pixels
print(f"Calculated Mean: {mean}")



100%|██████████| 170M/170M [00:15<00:00, 11.3MB/s]


Calculated Mean: tensor([0.4914, 0.4822, 0.4465])


In [ ]:
'''
RGB 평균값 각 픽셀에서 뺌
stride : 1
3 * 3 conv.layer padding 1 pixel -> spatial resolution preserved
'''
# 최종 Transform 정의 (정규화 포함)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=[1.0,1.0,1.0])
])

# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=mean.tolist(), std=[0.2023,0.1994,0.2010])
# ])
# cifar_std  = [0.2023, 0.1994, 0.2010]


# 학습/검증 데이터셋 로드
train_dataset = datasets.CIFAR10(root=data_dir, train=True, download=False, transform=transform)
val_dataset   = datasets.CIFAR10(root=data_dir, train=False, download=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Train : {len(train_loader)}, Test : {len(val_loader)}")

Train : 782, Test : 157


### Modeling

In [ ]:
def conv_layer_2(dim_in, dim_out) :
    model = nn.Sequential(
        nn.Conv2d(in_channels=dim_in, out_channels=dim_out, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=dim_out, out_channels=dim_out, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2) # 2*2맞는지?
    )
    return model

def conv_layer_3(dim_in, dim_out) :
    model = nn.Sequential(
        nn.Conv2d(in_channels=dim_in, out_channels=dim_out, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=dim_out, out_channels=dim_out, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=dim_out, out_channels=dim_out, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )
    return model

In [ ]:
class CustomVGG(nn.Module):
    def __init__(self, num_classes):
        super(CustomVGG, self).__init__()
        # convolution
        self.conv1 = conv_layer_2(3, 64) # (3, 224, 224) -> (64, 224, 224) -> (64, 224, 224) -> pooling
        self.conv2 = conv_layer_2(64,128) # (64, 112, 112) -> (128, 112, 112) -> (128, 112, 112) -> pooling

        self.conv3 = conv_layer_3(128, 256) # (128, 56, 56) -> (256, 56, 56) ->(256, 56, 56) -> (256, 56, 56) -> p
        self.conv4 = conv_layer_3(256, 512) # (256, 28, 28) -> (512, 28, 28) ->(512, 28, 28) -> (512, 28, 28) -> p
        self.conv5 = conv_layer_3(512, 512) # (512, 14, 14) -> (512, 14, 14) ->(512, 14, 14) -> (512, 14, 14) -> p

        # maxpooling
        # Input: (N, 512, 7, 7) -> flattened size: 512 * 7 * 7 = 25088
        self.fc1 = nn.Linear(25088, 4096)
        self.relu = nn.ReLU()
        # self.fc2 = nn.Linear(4096, 4096)
        # self.fc3 = nn.Linear(4096, num_classes)
        self.fc2 = nn.Linear(4096, 256)
        self.fc3 = nn.Linear(256, num_classes)
        self.softmax = nn.Softmax()
        # dropout = 0.5
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        # x = self.softmax(x) # CrossEntropyLoss 는 내부적으로 softmax 포함
        return x

custom_model = CustomVGG(num_classes=10).to(device)

In [ ]:
'''
Optimizer : SGD : batch sze = 256, momentum = 0.9, weight_decay = 5 * 10^-4 ,
            Learning rate schedule :

# Loss function : CrossEntopyLoss
'''

criterion = nn.CrossEntropyLoss()
# model.parameters() : 각 레이어의 가중치, 편향을 저장하는 모델파라미터

optimizer = optim.SGD(
    custom_model.parameters(),
    lr=0.01,              # 초깃값 0.01
    momentum=0.9,
    weight_decay=5e-4
)



In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 스케쥴러로
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=5, min_lr=1e-7)

#%%
from tqdm import tqdm

# -------------------------
# 4. 학습 루프
# -------------------------
EPOCHS = 10
for epoch in range(EPOCHS):
    custom_model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = custom_model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()

        '''
        # 1. 특정 레이어의 기울기 크기 확인 (L2 Norm)
      # conv1의 첫 번째 컨볼루션 레이어의 가중치 기울기 확인 (Weight)
        conv1_grad_norm = custom_model.conv1[0].weight.grad.norm().item()
        print(f"\n[DIAG] Conv1 Grad Norm: {conv1_grad_norm:.6f}")

    #   2. NaN 또는 Inf 포함 여부 확인 (전체 모델)
        has_nan_inf = False
        for name, param in custom_model.named_parameters():
            if param.grad is not None:
                if torch.isnan(param.grad).any() or torch.isinf(param.grad).any():
                    print(f"ERROR: {name} Gradient is NaN/Inf")
                    has_nan_inf = True
                    break

        if has_nan_inf:
            # NaN/Inf가 발견되면 학습을 멈추고 문제 진단을 시작해야 합니다.
            print("기울기 폭발")
            # return # 디버깅을 위해 학습을 멈추고 싶다면 주석 해제
        '''


        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    print(f"Train Loss: {running_loss/len(train_loader):.4f}, Acc: {train_acc*100:.2f}%")

    # -------------------------
    # 5. Single-scale Evaluation
    # -------------------------
    custom_model.eval()
    val_loss = 0.0
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = custom_model(imgs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f"Val Loss: {val_loss/len(val_loader):.4f}, Acc: {val_acc*100:.2f}%")
    # ---- 3️⃣ Scheduler update ----
    scheduler.step(val_acc)  # ← 이게 바로 여기 들어감!

    # ---- 4️⃣ 현재 학습률 출력 ----
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr:.6f}")

Epoch 1/10 [Train]: 100%|██████████| 782/782 [02:31<00:00,  5.18it/s]


Train Loss: 2.3030, Acc: 9.77%


Epoch 1/10 [Val]: 100%|██████████| 157/157 [00:18<00:00,  8.56it/s]


Val Loss: 2.3030, Acc: 10.00%
Current Learning Rate: 0.010000


Epoch 2/10 [Train]: 100%|██████████| 782/782 [02:30<00:00,  5.19it/s]


...

Val Loss: 2.3026, Acc: 10.00%
Current Learning Rate: 0.001000


Epoch 10/10 [Train]: 100%|██████████| 782/782 [02:30<00:00,  5.18it/s]


Train Loss: 2.3027, Acc: 9.84%


Epoch 10/10 [Val]: 100%|██████████| 157/157 [00:18<00:00,  8.47it/s]

Val Loss: 2.3026, Acc: 10.00%
Current Learning Rate: 0.001000


## custom VGG

## pretrained VGG

In [ ]:
base_model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
base_model.classifier = nn.Sequential(*list(base_model.classifier.children())[:-1])

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 154MB/s]


In [ ]:
class CustomVGG16(nn.Module):
    def __init__(self, num_classes):
        super(CustomVGG16, self).__init__()
        self.base_model = base_model
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(512, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

model = CustomVGG16(num_classes=10).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

Epoch 1, Loss: 1.7860223554894137
Epoch 2, Loss: 1.6297607063637365
Epoch 3, Loss: 1.5904177191007474
Epoch 4, Loss: 1.5740404487265955
Epoch 5, Loss: 1.5605740815477298
Epoch 6, Loss: 1.5534923773287508
Epoch 7, Loss: 1.5442786960650587
Epoch 8, Loss: 1.5331284236115263
Epoch 9, Loss: 1.5306233621924126
Epoch 10, Loss: 1.5241388696843705


In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 90.75%
